Item Classification — Fast/Slow Moving, Volume, Price, Profit Margin
Classifies each battery item into High/Low tiers per metric, using user-adjustable percentile thresholds.

**Input**: silver/erp/battery/battery_clean_live.json
**Output**: gold/erp/battery/analysis/item_classification.xlsx

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

Load silver

In [0]:
silver = read_silver(blob_service, "live/battery/battery_clean_live.json")
silver["posting_date"] = pd.to_datetime(silver["posting_date"])
print(f"Silver: {silver.shape}")

Compute per-item metrics

In [0]:
item_agg = silver.groupby("item_no").agg(
    total_units=("net_units", "sum"),
    total_revenue=("salesAmountActual", "sum"),
    total_profit=("profit", "sum"),
    first_sale=("posting_date", "min"),
    last_sale=("posting_date", "max"),
).reset_index()

item_agg["active_weeks"] = ((item_agg["last_sale"] - item_agg["first_sale"]).dt.days / 7).clip(lower=1)
item_agg["velocity_units_per_week"] = item_agg["total_units"] / item_agg["active_weeks"]
item_agg["avg_unit_price"] = item_agg["total_revenue"] / item_agg["total_units"].replace(0, pd.NA)

# Flag zero-revenue items separately — margin is undefined for these, not just "low"
item_agg["has_zero_revenue"] = item_agg["total_revenue"] == 0
item_agg["gross_margin_pct"] = item_agg.apply(
    lambda row: (row["total_profit"] / row["total_revenue"]) * 100 if row["total_revenue"] != 0 else pd.NA,
    axis=1
)

item_agg = item_agg[item_agg["total_units"] > 0].copy()

print(f"Items with zero revenue but real activity: {item_agg['has_zero_revenue'].sum()}")
print(item_agg[item_agg["has_zero_revenue"]][["item_no", "total_units", "total_profit"]])

Widgets for user-adjustable percentile thresholds

In [0]:
dbutils.widgets.text("volume_percentile", "70")
dbutils.widgets.text("velocity_percentile", "70")
dbutils.widgets.text("price_percentile", "70")
dbutils.widgets.text("margin_percentile", "70")

volume_pct = float(dbutils.widgets.get("volume_percentile"))
velocity_pct = float(dbutils.widgets.get("velocity_percentile"))
price_pct = float(dbutils.widgets.get("price_percentile"))
margin_pct = float(dbutils.widgets.get("margin_percentile"))

print(f"Thresholds - Volume: {volume_pct}th pct, Velocity: {velocity_pct}th pct, Price: {price_pct}th pct, Margin: {margin_pct}th pct")

Classify each metric

In [0]:
def classify(series, percentile):
    valid = series.dropna()
    cutoff = valid.quantile(percentile / 100)
    def label(x):
        if pd.isna(x):
            return "N/A"
        return "High" if x > cutoff else "Low"
    return series.apply(label), cutoff

item_agg["volume_tier"], volume_cutoff = classify(item_agg["total_units"], volume_pct)
item_agg["velocity_tier"], velocity_cutoff = classify(item_agg["velocity_units_per_week"], velocity_pct)
item_agg["price_tier"], price_cutoff = classify(item_agg["avg_unit_price"], price_pct)
item_agg["margin_tier"], margin_cutoff = classify(item_agg["gross_margin_pct"], margin_pct)

print(f"Volume cutoff: {volume_cutoff:.0f} units")
print(f"Velocity cutoff: {velocity_cutoff:.2f} units/week")
print(f"Price cutoff: {price_cutoff:.2f}")
print(f"Margin cutoff: {margin_cutoff:.2f}%")

print("\nTier counts:")
for col in ["volume_tier", "velocity_tier", "price_tier", "margin_tier"]:
    print(f"\n{col}:")
    print(item_agg[col].value_counts())

Rename velocity tier to fast/slow moving for readability, finalize table

In [0]:
item_agg["movement_tier"] = item_agg["velocity_tier"].map({"High": "Fast Moving", "Low": "Slow Moving"})

final_table = item_agg[[
    "item_no", "total_units", "volume_tier",
    "velocity_units_per_week", "movement_tier",
    "avg_unit_price", "price_tier",
    "gross_margin_pct", "margin_tier",
    "has_zero_revenue",
    "first_sale", "last_sale"
]].sort_values("total_units", ascending=False)

print(final_table)

Save to Excel for review

In [0]:
buffer = io.BytesIO()
final_table.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/analysis/item_classification.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved item_classification.xlsx")